In [1]:
print("Hello from R")

library(spacexr)
library(Matrix)
library(doParallel)

[1] "Hello from R"


Loading required package: foreach

Loading required package: iterators

Loading required package: parallel



In [4]:
library(anndata)
library(Matrix)
library(spacexr)

# -------------------------
# Step 1: Load AnnData
# -------------------------
# adata <- read_h5ad("adata_cside_xenium.h5ad")
adata <- read_h5ad("adata_cside_proseg.h5ad")

# Extract and transpose counts
counts <- t(adata$layers[["counts"]])  # genes x cells
genes <- rownames(adata$var)
barcodes <- rownames(adata$obs)
coords <- as.data.frame(adata$obsm[["spatial"]])
rownames(coords) <- barcodes
nUMI <- Matrix::colSums(counts)

# cell_types <- adata$obs$cell_type_coarse
cell_types <- adata$obs$cell_type
names(cell_types) <- barcodes

# Define group_labels from .obs$group
group_labels <- adata$obs$group
names(group_labels) <- barcodes

reference <- Reference(counts, cell_types, nUMI, min_UMI = 1)


# Load filtered AnnData with only G1 and G2
adata_G1G2 <- read_h5ad("adata_G1_G2_proseg.h5ad")
# adata_G1G2 <- read_h5ad("adata_G1_N1.h5ad")

# Extract counts and spatial
counts_G1G2 <- t(adata_G1G2$layers["counts"])  # genes x barcodes
coords_G1G2 <- as.data.frame(adata_G1G2$obsm["spatial"])
rownames(coords_G1G2) <- rownames(adata_G1G2$obs)
nUMI_G1G2 <- Matrix::colSums(counts_G1G2)

# Build new SpatialRNA object
puck_G1G2 <- SpatialRNA(coords_G1G2, counts_G1G2, nUMI_G1G2)


# -------------------------
myRCTD <- create.RCTD(puck_G1G2, reference, max_cores = 2)
# -------------------------
# Extract .obsm slot
weights <- adata_G1G2$obsm[["cell_type_onehot"]]
weights <- as(Matrix(weights, sparse = TRUE), "dgCMatrix")
storage.mode(weights@x) <- "double"              # enforce numeric values
# Set rownames from adata$obs_names (pixel/cell names)
rownames(weights) <- rownames(adata_G1G2$obs)

# Set colnames from adata$uns["cell_type_onehot_columns"]
colnames(weights) <- adata_G1G2$uns[["cell_type_onehot_columns"]]

myRCTD <- import_weights(myRCTD, weights)

Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 24562

process_cell_type_info: number of genes in reference: 313




          B_Cells      CD4+_T_Cells      CD8+_T_Cells              DCIS 
              522              1288              1535               406 
      Endothelial         IRF7+_DCs    Invasive_Tumor     Macrophages_1 
             5941                36              3984              2199 
    Macrophages_2        Mast_Cells     Myoepi_ACTA2+ Perivascular-Like 
              432                43               864               465 
          Stromal         Unlabeled 
             6678               169 


End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: B_Cells found DE genes: 40

get_de_genes: CD4+_T_Cells found DE genes: 47

get_de_genes: CD8+_T_Cells found DE genes: 41

get_de_genes: DCIS found DE genes: 45

get_de_genes: Endothelial found DE genes: 45

get_de_genes: IRF7+_DCs found DE genes: 29

get_de_genes: Invasive_Tumor found DE genes: 45

get_de_genes: Macrophages_1 found DE genes: 36

get_de_genes: Macrophages_2 found DE genes: 38

get_de_genes: Mast_Cells found DE genes: 24

get_de_genes: Myoepi_ACTA2+ found DE genes: 36

get_de_genes: Perivascular-Like found DE genes: 43

get_de_genes: Stromal found DE genes: 29

get_de_genes: Unlabeled found DE genes: 48

get_de_genes: total DE genes: 294

create.RCTD: getting platform effect normalization differentially expressed genes: 

get_de_genes: B_Cells found DE genes: 50

get_de_genes: CD4+_T_Cells found DE genes: 53

get_de_genes: CD8+_T_Cells found DE genes: 45

get_de_g

In [6]:
print(dim(weights))

[1] 5941   14


In [7]:
# -------------------------
# Step: Build explanatory variable
# -------------------------
# ✅ Correct barcodes (columns of counts, not rows!)
barcodes <- colnames(myRCTD@spatialRNA@counts)

# Group labels (also ensure character rownames)
group_labels_G1G2 <- adata_G1G2$obs$group
names(group_labels_G1G2) <- as.character(rownames(adata_G1G2$obs))

# Explanatory variable
expl_var <- as.numeric(group_labels_G1G2[barcodes] == "G1")
names(expl_var) <- barcodes

# Double check common barcodes
cat("✅ Common barcodes:", length(intersect(names(expl_var), rownames(puck_G1G2@counts))), "\n")


# Filter weights to G1/G2 barcodes
weights_G1G2 <- weights
# -------------------------
# Step: Filter genes using puck_G1G2 (spatial counts)
# -------------------------

gene_list <- spacexr:::filter_genes(puck_G1G2)

cat("✅ Number of genes after filtering:", length(gene_list), "\n")

# -------------------------
# Step: Run CSIDE on G1 vs G2
# -------------------------

myRCTD <- run.CSIDE.single(
  myRCTD,
  explanatory.variable = expl_var,
  doublet_mode = FALSE,
  weight_threshold = 0.75,
  # test_genes_sig = gene_list,
  logs = TRUE,
  # cell_type_threshold = 20,
  log_fc_thresh=0.0,
  fdr=0.05,
)

# Save results
saveRDS(myRCTD, file = "myRCTD_CSIDE_G1_vs_G2_proseg.rds")

✅ Common barcodes: 0 


filter_genes: filtering genes based on threshold = 5e-05



✅ Number of genes after filtering: 296 


run.CSIDE.single: filtered out cell types: c(B_Cells = 1, `CD4+_T_Cells` = 2, `CD8+_T_Cells` = 3, DCIS = 4, `IRF7+_DCs` = 6, Invasive_Tumor = 7, Macrophages_1 = 8, Macrophages_2 = 9, Mast_Cells = 10, `Myoepi_ACTA2+` = 11, `Perivascular-Like` = 12, Stromal = 13, Unlabeled = 14) due to not having sufficiently many pixels with explanatory.value on either side of medv = 0.5. Please note that it is required to have on either side of medv at least (one half of cell_type_threshold) = 62.5 pixels.

choose_cell_types: detected the following cell types occuring at least cell_type_threshold times: Endothelial


run.CSIDE.general: running CSIDE with cell types Endothelial

run.CSIDE.general: configure params_to_test = 2, 

filter_genes: filtering genes based on threshold = 5e-05

set_global_Q_all: begin

set_global_Q_all: finished



In [8]:
names(myRCTD@de_results$sig_gene_list)

[1] "Endothelial"

In [9]:
cell_type <- "Endothelial"  # or "2" if you're using numeric indices
results_de <- myRCTD@de_results$sig_gene_list[[cell_type]]

# View top results
head(results_de)

,Z_score,log_fc,se,paramindex_best,conv,p_val,mean_0,mean_1,sd_0,sd_1
,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ADH1B,36.60773,-4.3095640,0.11772279,2,TRUE,2.154774e-293,-5.648423,-9.957987,0.07214489,0.09302564
KDR,26.66251,0.8771044,0.03289655,2,TRUE,1.281714e-156,-4.764310,-3.887206,0.02961652,0.01431938
ESM1,25.20492,3.2041274,0.12712311,2,TRUE,3.538147e-140,-8.974097,-5.769969,0.12083346,0.03949127
CAVIN2,23.89205,-1.1813154,0.04944388,2,TRUE,3.704936e-126,-5.048213,-6.229528,0.04028358,0.02866932
PDK4,23.52550,-1.0078507,0.04284078,2,TRUE,2.237080e-122,-4.186480,-5.194331,0.03623184,0.02286016
TIMP4,22.64099,-2.0710663,0.09147420,2,TRUE,1.711599e-113,-6.571726,-8.642792,0.06811208,0.06105959


In [10]:
sig_genes <- rownames(results_de)[
  results_de$log_fc > 0 & results_de$p_val < 0.05
]

sig_genes

[1] "KDR"      "ESM1"     "TCIM"     "RAMP2"    "CCND1"    "CLEC14A" 
 [7] "PRDM1"    "POLR2J3"  "ZEB1"     "VOPP1"    "FBLIM1"   "IL3RA"   
[13] "JUP"      "PCLAF"    "SNAI1"    "CD93"     "SOX18"    "DUSP5"   
[19] "TCF4"     "TRAF4"    "TOP2A"    "CDC42EP1" "SEC24A"   "ERBB2"   
[25] "FOXA1"    "CENPF"    "NDUFA4L2" "CCPG1"    "SQLE"     "MMP2"    
[31] "ADGRE5"   "ADAM9"    "AVPR1A"   "TUBA4A"   "CTTN"     "FOXC2"   
[37] "CD9"      "STC1"     "PDGFRB"   "SVIL"     "MKI67"    "EIF4EBP1"
[43] "PELI1"    "OXTR"     "SMS"      "DNTTIP1"  "EDNRB"    "FAM107B" 
[49] "NARS"     "CCDC6"    "MDM2"     "ACTA2"    "RUNX1"    "DAPK3"   
[55] "LDHB"     "LGALSL"   "CXCR4"    "PTRHD1"   "TRAPPC3"  "MYLK"    
[61] "HDC"      "TFAP2A"   "QARS"     "SRPK1"    "GJB2"     "SCD"     
[67] "PECAM1"   "TMEM147"  "ZNF562"   "SERHL2"   "SERPINB9" "VWF"     
[73] "KRT7"     "ANKRD28"  "PTN"      "ABCC11"   "FASN"     "THAP2"   
[79] "LRRC15"

In [11]:
# Print in Python list format
cat("sig_genes = [", paste(sprintf('"%s"', sig_genes), collapse=", "), "]\n")

sig_genes = [ "KDR", "ESM1", "TCIM", "RAMP2", "CCND1", "CLEC14A", "PRDM1", "POLR2J3", "ZEB1", "VOPP1", "FBLIM1", "IL3RA", "JUP", "PCLAF", "SNAI1", "CD93", "SOX18", "DUSP5", "TCF4", "TRAF4", "TOP2A", "CDC42EP1", "SEC24A", "ERBB2", "FOXA1", "CENPF", "NDUFA4L2", "CCPG1", "SQLE", "MMP2", "ADGRE5", "ADAM9", "AVPR1A", "TUBA4A", "CTTN", "FOXC2", "CD9", "STC1", "PDGFRB", "SVIL", "MKI67", "EIF4EBP1", "PELI1", "OXTR", "SMS", "DNTTIP1", "EDNRB", "FAM107B", "NARS", "CCDC6", "MDM2", "ACTA2", "RUNX1", "DAPK3", "LDHB", "LGALSL", "CXCR4", "PTRHD1", "TRAPPC3", "MYLK", "HDC", "TFAP2A", "QARS", "SRPK1", "GJB2", "SCD", "PECAM1", "TMEM147", "ZNF562", "SERHL2", "SERPINB9", "VWF", "KRT7", "ANKRD28", "PTN", "ABCC11", "FASN", "THAP2", "LRRC15" ]


In [12]:
dim(results_de)

[1] 212  10

In [39]:
?run.CSIDE.single

run.CSIDE.single            package:spacexr            R Documentation

_R_u_n_s _C_S_I_D_E _o_n _a '_R_C_T_D' _o_b_j_e_c_t _w_i_t_h _a _s_i_n_g_l_e _e_x_p_l_a_n_a_t_o_r_y _v_a_r_i_a_b_l_e

_D_e_s_c_r_i_p_t_i_o_n:

     Identifies cell type specific differential expression (DE) as a
     function of the explanatory variable. The design matrix contains
     an intercept column and a column of the explanatory variable. Uses
     maximum likelihood estimation to estimate DE and standard errors
     for each gene and each cell type. Selects genes with significant
     nonzero DE.

_U_s_a_g_e:

     run.CSIDE.single(
       myRCTD,
       explanatory.variable,
       cell_types = NULL,
       cell_type_threshold = 125,
       gene_threshold = 5e-05,
       doublet_mode = T,
       weight_threshold = NULL,
       sigma_gene = T,
       PRECISION.THRESHOLD = 0.05,
       cell_types_present = NULL,
       fdr = 0.01,
       te